# RAG sobre papers académicos con NVIDIA NIM — mi copia (Bloque B + retos)

Pipeline: **PDFs → chunks → Qdrant → LCEL → evaluación**. Papers del corpus:

- `attention_paper.pdf` — *Attention Is All You Need* (Vaswani et al., 2017), arXiv 1706.03762
- `rag_paper.pdf` — *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks* (Lewis et al., 2020), arXiv 2005.11401
- `hnsw_paper.pdf` — *Efficient and robust ANN using HNSW graphs* (Malkov & Yashunin, 2016), arXiv 1603.09320 **(reto C2)**

Parámetros base del notebook: `chunk_size=900`, `chunk_overlap=120`, `k=5`, `score_threshold=0.25`.

**Modelos:** embeddings `nvidia/nemotron-3-embed-1b`, generación `nvidia/nemotron-3-super-120b-a12b`.

## Parte 0 · Instalación y API key

In [ ]:
%pip install -qU gradio langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client pypdf requests


In [ ]:
import os, json, time, getpass
from pathlib import Path
from typing import List, Dict

import requests
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, Filter, FieldCondition, MatchValue
from langchain_qdrant import QdrantVectorStore

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

# Selección robusta de modelos: evita el error 404 'Not found for account'
# eligiendo solo entre los modelos realmente habilitados para tu API key.
def _elegir(preferidos, disponibles):
    for p in preferidos:
        if p in disponibles:
            return p
    return disponibles[0] if disponibles else None

_chat = [m.id for m in ChatNVIDIA.get_available_models()]
_emb = [m.id for m in NVIDIAEmbeddings.get_available_models()]
GENERATION_MODEL = _elegir([
    "nvidia/nemotron-3-super-120b-a12b",
    "nvidia/llama-3.3-nemotron-super-49b-v1",
    "meta/llama-3.1-70b-instruct", "meta/llama-3.1-8b-instruct",
], _chat)
EMBEDDING_MODEL = _elegir([
    "nvidia/nemotron-3-embed-1b",
    "nvidia/llama-3.2-nv-embedqa-1b-v2", "nvidia/nv-embedqa-e5-v5", "NV-Embed-QA",
], _emb)
COLLECTION = "academic_papers_nvidia"
assert GENERATION_MODEL and EMBEDDING_MODEL, "Tu key no expone modelos: revisa Public API Endpoints en build.nvidia.com"
print(f"✅ Configuración lista — gen: {GENERATION_MODEL} | emb: {EMBEDDING_MODEL}")


## Parte 1 · Descargar los PDFs

El diccionario `PAPERS` incluye el tercer paper de HNSW (**reto C2**).

In [ ]:
PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
    # Reto C2: paper original de HNSW.
    "hnsw_paper.pdf": "https://arxiv.org/pdf/1603.09320",
}

papers_dir = Path("papers"); papers_dir.mkdir(exist_ok=True)
for name, url in PAPERS.items():
    dest = papers_dir / name
    if dest.exists():
        print(f"• {name} ya existe"); continue
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    r.raise_for_status(); dest.write_bytes(r.content)
    print(f"✅ {name} descargado ({len(r.content)//1024} KB)")


## Parte 2 · PDFs → chunks

In [ ]:
def cargar_chunks(chunk_size=900, chunk_overlap=120) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " "],
    )
    all_docs: List[Document] = []
    for name in PAPERS:
        pages = PyPDFLoader(str(papers_dir / name)).load()
        full = "\n".join(p.page_content for p in pages)
        for i, text in enumerate(splitter.split_text(full)):
            all_docs.append(Document(
                page_content=text,
                metadata={"source": name, "chunk_index": i,
                          "title": name.replace("_", " ").replace(".pdf", "")},
            ))
    return all_docs

all_docs = cargar_chunks()
print(f"✅ {len(all_docs)} chunks desde {len(PAPERS)} papers")


## Parte 3 · Indexar en Qdrant

In [ ]:
embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")

def construir_vectorstore(docs, client=None):
    dim = len(embeddings.embed_query("prueba"))
    client = client or QdrantClient(":memory:")
    if client.collection_exists(COLLECTION):
        client.delete_collection(COLLECTION)
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
    )
    vs = QdrantVectorStore(client=client, collection_name=COLLECTION, embedding=embeddings)
    vs.add_documents(docs)
    return vs

vectorstore = construir_vectorstore(all_docs)
print(f"✅ {len(all_docs)} chunks indexados en Qdrant")


## Parte 4 · Cadena RAG con LCEL

In [ ]:
llm = ChatNVIDIA(model=GENERATION_MODEL, temperature=0.2, top_p=0.95, max_completion_tokens=700)

SYSTEM_PROMPT = (
    "Eres un asistente experto en papers de aprendizaje automático. Responde SOLO con "
    "información del contexto. Si el contexto no alcanza, di que no está en los papers. "
    "Cita el título del paper. Responde en el idioma de la pregunta."
)

def construir_cadena(vs, k=5, score_threshold=0.25, system_prompt=SYSTEM_PROMPT, search_type="similarity_score_threshold", search_kwargs=None):
    if search_kwargs is None:
        search_kwargs = {"k": k, "score_threshold": score_threshold}
    retriever = vs.as_retriever(search_type=search_type, search_kwargs=search_kwargs)
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])
    def format_docs(items):
        return "\n\n---\n\n".join(f"[{d.metadata.get('source')}] {d.page_content}" for d in items)
    setup = RunnableParallel(question=RunnablePassthrough(), documents=retriever)
    answer = ({"context": lambda x: format_docs(x["documents"]), "question": lambda x: x["question"]}
              | prompt | llm | StrOutputParser())
    return setup.assign(answer=answer), retriever

rag_chain, retriever = construir_cadena(vectorstore)

def preguntar(q):
    r = rag_chain.invoke(q)
    fuentes = sorted({d.metadata["source"] for d in r["documents"]})
    return r["answer"], fuentes

print("✅ Cadena RAG lista")


## B1 · Quiz con ground truth

<details><summary>Q1–Q5 + Bonus con respuesta de referencia</summary>

- **Q1.** How many attention heads does the base Transformer use? — *GT: h = 8 heads, con d_k = d_v = 64.*
- **Q2.** What optimizer and learning-rate schedule trains the Transformer? — *GT: Adam con warmup y decaimiento ∝ 1/√step.*
- **Q3.** What are the two RAG model variants? — *GT: RAG-Sequence y RAG-Token.*
- **Q4.** Which retriever and index does the RAG paper use? — *GT: DPR (dense passage retriever) sobre Wikipedia con un índice FAISS (MIPS).*
- **Q5.** What is the dimensionality d_model of the base Transformer? — *GT: 512.*
- **Bonus.** How does the RAG paper relate to approximate nearest neighbor search? — *GT: usa FAISS/MIPS para recuperar pasajes; conecta con ANN/HNSW.*

</details>

In [ ]:
QUIZ = [
    ("Q1", "How many attention heads does the base Transformer use?"),
    ("Q2", "What optimizer and learning-rate schedule trains the Transformer?"),
    ("Q3", "What are the two RAG model variants?"),
    ("Q4", "Which retriever and index does the RAG paper use?"),
    ("Q5", "What is the dimensionality d_model of the base Transformer?"),
    ("Bonus", "How does the RAG paper relate to approximate nearest neighbor search?"),
]
for qid, q in QUIZ:
    ans, fuentes = preguntar(q)
    print(f"\n[{qid}] {q}\n → {ans}\n   fuentes: {fuentes}")


## B2 · Experimentos (chunk_size, k, score_threshold)

Funciones auxiliares + los 3 experimentos. **Ojo:** el experimento 1 reconstruye el índice 3 veces (3 llamadas a embeddings).

In [ ]:
GROUND_TRUTH = {
    "Q1": ["8", "eight"], "Q2": ["adam", "warmup"], "Q3": ["rag-sequence", "rag-token"],
    "Q4": ["dpr", "faiss"], "Q5": ["512"],
}

def evaluar_quiz(chain):
    aciertos = 0
    for qid, q in QUIZ:
        if qid == "Bonus":
            continue
        ans = chain.invoke(q)["answer"].lower()
        ok = any(tok in ans for tok in GROUND_TRUTH[qid])
        aciertos += ok
    return aciertos

def experimento(chunk_size=None, k=5, score_threshold=0.25, vs=None):
    if vs is None:
        docs = cargar_chunks(chunk_size=chunk_size, chunk_overlap=int(chunk_size*0.13))
        vs = construir_vectorstore(docs)
        n_chunks = len(docs)
    else:
        n_chunks = None
    chain, _ = construir_cadena(vs, k=k, score_threshold=score_threshold)
    return n_chunks, evaluar_quiz(chain), vs

# Experimento 1 — chunk_size (3 construcciones)
print("== Exp 1: chunk_size ==")
vs_por_size = {}
for cs in (400, 900, 1500):
    n, aciertos, vs = experimento(chunk_size=cs)
    vs_por_size[cs] = vs
    print(f"chunk_size={cs}: {n} chunks, {aciertos}/5 aciertos")

# Experimento 2 — k (reutiliza el índice de 900)
print("\n== Exp 2: k (índice 900) ==")
for k in (3, 5, 8):
    _, aciertos, _ = experimento(k=k, vs=vs_por_size[900])
    print(f"k={k}: {aciertos}/5 aciertos")

# Experimento 3 — score_threshold (reutiliza el índice de 900)
print("\n== Exp 3: score_threshold (índice 900) ==")
for st in (0.15, 0.25, 0.40):
    _, aciertos, _ = experimento(score_threshold=st, vs=vs_por_size[900])
    print(f"score_threshold={st}: {aciertos}/5 aciertos")


## B3 · Preguntas trampa (¿admite que no sabe?)

Tres preguntas plausibles cuya respuesta NO está en los papers.

In [ ]:
TRAMPA = [
    "What BLEU score does GPT-4 achieve on the WMT 2023 English-German task?",
    "How many parameters does Llama 3 70B use according to these papers?",
    "What is the training cost in dollars of the Transformer on TPUv5 pods?",
]
for q in TRAMPA:
    ans, fuentes = preguntar(q)
    print(f"\nP: {q}\n → {ans}\n   fuentes: {fuentes}")

# Prompt endurecido anti-alucinación (B3).
SYSTEM_PROMPT_ESTRICTO = (
    "Eres un asistente sobre papers de ML. Usa EXCLUSIVAMENTE el contexto. "
    "Si la respuesta no está literalmente en el contexto, responde SOLO: "
    "'No encuentro esa información en los papers indexados.' No uses conocimiento previo. "
    "No menciones modelos ni cifras que no aparezcan en el contexto. Cita el paper fuente."
)
rag_estricto, _ = construir_cadena(vectorstore, system_prompt=SYSTEM_PROMPT_ESTRICTO)
print("\n=== Con prompt estricto ===")
for q in TRAMPA:
    print(f"\nP: {q}\n → {rag_estricto.invoke(q)['answer']}")


## B4 · Preguntas en español sobre papers en inglés

In [ ]:
PARES = [
    ("Q1", "How many attention heads does the base Transformer use?",
            "¿Cuántas cabezas de atención usa el Transformer base?"),
    ("Q3", "What are the two RAG model variants?",
            "¿Cuáles son las dos variantes del modelo RAG?"),
    ("Q4", "Which retriever and index does the RAG paper use?",
            "¿Qué recuperador e índice usa el paper de RAG?"),
]
for qid, en, es in PARES:
    docs_en = [d.metadata["chunk_index"] for d in retriever.invoke(en)]
    docs_es = [d.metadata["chunk_index"] for d in retriever.invoke(es)]
    print(f"\n[{qid}]")
    print(f"  EN chunks: {docs_en}")
    print(f"  ES chunks: {docs_es}")
    print(f"  solapamiento: {len(set(docs_en) & set(docs_es))}/{len(docs_en)}")
    print(f"  ES → {preguntar(es)[0]}")


## B5 · Mi propio quiz con ground truth

<details><summary>Q6–Q8 escritas por mí (una cruza los dos papers)</summary>

- **Q6.** In the Transformer, what operation replaces recurrence to model dependencies? — *GT: self-attention (atención sobre la propia secuencia). Sección 3.2 / 4 de Attention Is All You Need.*
- **Q7.** In the RAG paper, is the document index trained end-to-end with the generator? — *GT: no; el codificador del documento y el índice se mantienen fijos, solo se ajusta el query encoder y el generador. Sección 2.* 
- **Q8 (cruza los dos).** Both the Transformer and RAG rely on attention; what extra component does RAG add on top of a seq2seq Transformer to inject external knowledge? — *GT: un recuperador denso (DPR) + índice de vecinos (FAISS) que aporta pasajes como contexto latente. Combina Attention (Q) y RAG (Q4).*

</details>

In [ ]:
MI_QUIZ = [
    ("Q6", "In the Transformer, what operation replaces recurrence to model dependencies?"),
    ("Q7", "In the RAG paper, is the document index trained end-to-end with the generator?"),
    ("Q8", "Both the Transformer and RAG rely on attention; what extra component does RAG add on top of a seq2seq Transformer to inject external knowledge?"),
]
for qid, q in MI_QUIZ:
    ans, fuentes = preguntar(q)
    print(f"\n[{qid}] {q}\n → {ans}\n   fuentes: {fuentes}")


## C2 · 3 preguntas sobre el paper de HNSW + 1 que cruza con RAG

In [ ]:
# GT verificado leyendo arXiv:1603.09320.
# H1: la capa de un elemento se escoge con una distribución exponencial decreciente
#     (l = floor(-ln(U)*mL)); H2: M es el número de conexiones por nodo por capa (grado);
#     H3: efConstruction controla el tamaño de la lista dinámica de candidatos al insertar.
HNSW_Q = [
    "In the HNSW paper, how is the layer of a new element chosen?",
    "In HNSW, what does the parameter M represent?",
    "In HNSW, what is the role of efConstruction?",
    # Cruce HNSW + RAG:
    "The RAG paper retrieves passages with a nearest-neighbor index; which graph-based ANN method could implement that index and how does it pick layers?",
]
for q in HNSW_Q:
    ans, fuentes = preguntar(q)
    print(f"\nP: {q}\n → {ans}\n   fuentes: {fuentes}")


## C4 · Evaluación automática de la recuperación (hit@k)

In [ ]:
import csv

def cargar_preguntas(ruta="eval/preguntas.csv"):
    with open(ruta, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def hit_at_k(vs, filas, k):
    retr = vs.as_retriever(search_type="similarity", search_kwargs={"k": k})
    hits = 0
    for r in filas:
        docs = retr.invoke(r["pregunta"])
        ok = any(r["palabra_clave"].lower() in d.page_content.lower()
                 and d.metadata["source"] == r["fuente_esperada"] for d in docs)
        hits += ok
    return hits / len(filas)

filas = cargar_preguntas()
print(f"{len(filas)} preguntas de evaluación")
tabla = []
for cs in (400, 900, 1500):
    vs = vs_por_size.get(cs) or construir_vectorstore(cargar_chunks(cs, int(cs*0.13)))
    h3, h5 = hit_at_k(vs, filas, 3), hit_at_k(vs, filas, 5)
    tabla.append((cs, h3, h5))
    print(f"chunk_size={cs}: hit@3={h3:.2f}  hit@5={h5:.2f}")


## C5 · MMR frente a similitud pura

In [ ]:
retr_sim = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})
retr_mmr = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5})

for q in ["What are the two RAG model variants?",
          "Describe the Transformer architecture components.",
          "How does HNSW search the graph?"]:
    print(f"\n### {q}")
    print("SIMILITUD:", [(d.metadata['source'], d.metadata['chunk_index']) for d in retr_sim.invoke(q)])
    print("MMR      :", [(d.metadata['source'], d.metadata['chunk_index']) for d in retr_mmr.invoke(q)])


## C3 · App Gradio para los papers con filtro por paper

In [ ]:
import gradio as gr

FILTRO_FUENTE = {
    "Todos": None,
    "Attention Is All You Need": "attention_paper.pdf",
    "Retrieval-Augmented Generation": "rag_paper.pdf",
    "HNSW (Malkov & Yashunin)": "hnsw_paper.pdf",
}

def recuperar_con_filtro(question, fuente, k=5):
    qfilter = None
    if fuente:
        qfilter = Filter(must=[FieldCondition(key="metadata.source", match=MatchValue(value=fuente))])
    return vectorstore.similarity_search_with_score(question, k=k, filter=qfilter)

def chat_papers(question, seleccion):
    fuente = FILTRO_FUENTE.get(seleccion)
    scored = recuperar_con_filtro(question, fuente)
    if not scored:
        return "Sin resultados para ese filtro.", ""
    contexto = "\n\n---\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d, _ in scored)
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])
    answer = (prompt | llm | StrOutputParser()).invoke({"context": contexto, "question": question})
    fuentes = "\n".join(f"- **{d.metadata['source']}** (chunk {d.metadata['chunk_index']}) · score `{s:.3f}`" for d, s in scored)
    return answer, fuentes

with gr.Blocks(title="RAG papers con filtro") as demo_papers:
    gr.Markdown("# RAG sobre papers · filtro por documento")
    dd = gr.Dropdown(list(FILTRO_FUENTE), value="Todos", label="Filtrar por paper")
    q = gr.Textbox(label="Pregunta", lines=2)
    btn = gr.Button("Preguntar", variant="primary")
    out = gr.Markdown(label="Respuesta")
    src = gr.Markdown(label="Fuentes")
    btn.click(chat_papers, inputs=[q, dd], outputs=[out, src])

# demo_papers.launch(share=True)  # descomenta en Colab
print("✅ App de papers con filtro lista (descomenta launch en Colab)")


## C6 · Persistencia del índice en disco

Cambia `QdrantClient(":memory:")` por un cliente con ruta local. Primera corrida indexa; tras reiniciar el runtime, reabre sin reindexar.

In [ ]:
client_disco = QdrantClient(path="./qdrant_data")

t0 = time.time()
if client_disco.collection_exists(COLLECTION):
    vs_disco = QdrantVectorStore(client=client_disco, collection_name=COLLECTION, embedding=embeddings)
    modo = "REABIERTO (sin reindexar)"
else:
    vs_disco = construir_vectorstore(all_docs, client=client_disco)
    modo = "INDEXADO (primera vez)"
print(f"{modo} en {time.time()-t0:.2f} s")

chain_disco, _ = construir_cadena(vs_disco)
print(chain_disco.invoke("How many attention heads does the base Transformer use?")["answer"])


## Preguntas de reflexión (R1)

Las respuestas desarrolladas están en `RESPUESTAS.md`, sección **Reflexión**.

1. ¿Qué ocurrió cuando el contexto no tenía información suficiente? ¿Reconoció la limitación o inventó?
2. ¿Qué fragmentos se recuperaron para la pregunta bonus? ¿Aparecieron ambos papers?
3. ¿Qué efecto tuvo modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?
4. ¿Por qué conviene separar el modelo de embeddings del generativo?
5. ¿Qué cambiarías para llevar esto a producción?